In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN

df = pd.read_csv("../data/kmean_cluster_data05.csv")


df.head()

,customer_id,age,annual_income,months_active,avg_monthly_spend,purchase_frequency,avg_order_value,discount_usage_rate,return_rate,browsing_time_minutes,support_interactions,payment_method,region,customer_segment,estimated_annual_spend,purchase_value_index,engagement_score,discount_dependency,customer_value_score,Cluster
0,33554,53,100473.211709,63,121.430565,0.817268,66.820403,0.117256,0.023144,77.298393,2.0,Card,Semi-Urban,Occasional,1457.166781,54.610162,63.173486,0.117256,99.241287,1
1,9428,54,54730.644845,67,572.552674,3.176551,137.087449,0.261647,0.429054,92.132565,2.0,Wallet,Urban,Occasional,6870.632088,435.465225,292.663757,0.261647,1818.742563,3
2,200,44,58268.121079,57,266.593896,2.713168,71.796888,0.284785,0.011854,155.194768,1.0,UPI,Rural,Occasional,3199.126755,194.797008,421.069457,0.284785,723.313990,0
3,12448,54,64829.795654,40,691.452358,5.553977,105.501185,0.104832,0.399686,113.917756,0.0,Wallet,Rural,High_Value,8297.428299,585.951173,632.696615,0.104832,3840.310600,3
4,39490,28,27431.467873,15,832.664792,1.348389,354.568534,0.409204,0.039517,50.123656,1.0,Card,Semi-Urban,Occasional,9991.977507,478.096258,67.586178,0.409204,1122.755920,2


In [2]:
import pickle

with open("../data/processed_variables.pkl", "rb") as f:
    data = pickle.load(f)

df = data["df"]
features = data["features"]
x_scaled = data["x_scaled"]
x_scaled_df = data["x_scaled_df"]
scaler = data["scaler"]
final_kmeans = ["final_kmeans"]

C:\Users\manya\AppData\Roaming\Python\Python314\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\manya\AppData\Roaming\Python\Python314\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator KMeans from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [4]:
# from sklearn.cluster import DBSCAN

dbscan = DBSCAN(
    eps=0.5,
    min_samples=5
)

dbscan_labels = dbscan.fit_predict(x_scaled)

df["DBSCAN_Cluster"] = dbscan_labels

In [5]:
from sklearn.metrics import silhouette_score
results = []

for eps in [0.3, 0.5, 0.7, 1.0]:

    for min_samples in [3, 5, 10]:

        dbscan = DBSCAN(
            eps=eps,
            min_samples=min_samples
        )

        labels = dbscan.fit_predict(x_scaled)

        unique_labels = set(labels)

        if len(unique_labels) > 1:

            score = silhouette_score(x_scaled, labels)

            results.append({
                "eps": eps,
                "min_samples": min_samples,
                "silhouette_score": score,
                "clusters": len(unique_labels)
            })

dbscan_results = pd.DataFrame(results)

dbscan_results.sort_values(
    "silhouette_score",
    ascending=False
).head()

,eps,min_samples,silhouette_score,clusters
3,1.0,10,-0.310570,13
0,0.7,3,-0.452051,73
2,1.0,5,-0.467882,209
1,1.0,3,-0.493933,1102


In [6]:
df.to_csv('../data/db_scanned_data06.csv',index=False)

In [7]:
import pickle

with open("../data/processed_variables.pkl", "wb") as f:
    pickle.dump({
        "df": df,
        "features": features,
        "x_scaled":x_scaled,
        "x_scaled_df":x_scaled_df,
        "scaler":scaler,
        "final_kmeans":final_kmeans,
        "dbscan_results":dbscan_results

    }, f)